In [1]:
# importing libraries

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import duckdb
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
import math
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

### Import Datasets

In [2]:
# reading datasets and combining them

df_cat = pd.read_csv('data/category_tree.csv')
df_event = pd.read_csv('data/events.csv')
df_item_prop_1 = pd.read_csv('data/item_properties_part1.csv')
df_item_prop_2 = pd.read_csv('data/item_properties_part2.csv')

# combining the both item prop dfs
df_item_prop = pd.concat([df_item_prop_1, df_item_prop_2], ignore_index=True)

del df_item_prop_1, df_item_prop_2

print(df_cat.shape, df_event.shape, df_item_prop.shape)

(1669, 2) (2756101, 5) (20275902, 4)


### Preprocess Datasets

In [3]:
# setting the parent to cat_id itself wherever nulls

# replacing the NaNs in the parentid 
print('before: ')
print(df_cat.isna().sum())
df_cat.loc[df_cat.parentid.isnull(),'parentid'] = df_cat.loc[df_cat.parentid.isnull(), 'categoryid']
print('After: ')
print(df_cat.isna().sum())

print("----------------------------------------------------")

# dropping duplicates from df_event
print('before: ')
print(df_event.duplicated().sum())
df_event.drop_duplicates(inplace=True)
print('After: ')
print(df_event.duplicated().sum())

print("----------------------------------------------------")

# replacing the timestamp to proper form
print('before: ')
print(df_event.timestamp[0:2])
print(df_item_prop.timestamp[0:2])

df_event['timestamp'] = pd.to_datetime(df_event['timestamp'], unit='ms') #.dt.strftime('%Y-%m-%d %H:%M:%S')
df_item_prop['timestamp'] = pd.to_datetime(df_item_prop['timestamp'], unit='ms') #.dt.strftime('%Y-%m-%d %H:%M:%S')

print('After: ')
print(df_event.timestamp[0:2])
print(df_item_prop.timestamp[0:2])

before: 
categoryid     0
parentid      25
dtype: int64
After: 
categoryid    0
parentid      0
dtype: int64
----------------------------------------------------
before: 
460
After: 
0
----------------------------------------------------
before: 
0    1433221332117
1    1433224214164
Name: timestamp, dtype: int64
0    1435460400000
1    1441508400000
Name: timestamp, dtype: int64
After: 
0   2015-06-02 05:02:12.117
1   2015-06-02 05:50:14.164
Name: timestamp, dtype: datetime64[ms]
0   2015-06-28 03:00:00
1   2015-09-06 03:00:00
Name: timestamp, dtype: datetime64[ms]


### Sessionization of events

In [4]:
sessionization_query = """
WITH ordered AS (
    select 
    *, 
    LAG(timestamp) OVER (
            PARTITION BY visitorid
            ORDER BY sequence_no
        ) AS prev_ts,
    from (
    SELECT
        *,
        row_number() OVER (
            PARTITION BY visitorid
            ORDER BY timestamp, itemid, coalesce(transactionid, -1)
        ) AS sequence_no
    FROM df_event
    ) t1
),

flagged AS (
    SELECT
        *,
        CASE
            WHEN prev_ts IS NULL THEN 1
            WHEN timestamp - prev_ts >= INTERVAL '30 minutes' THEN 1
            ELSE 0
        END AS is_new_session
    FROM ordered
),

sessionized AS (
    SELECT
        *,
        SUM(is_new_session) OVER (
            PARTITION BY visitorid
            ORDER BY sequence_no
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS session_id
    FROM flagged
)

SELECT
    *,
    MIN(timestamp) OVER (PARTITION BY visitorid, session_id) AS start_st,
    MAX(timestamp) OVER (PARTITION BY visitorid, session_id) AS end_st,
    ROW_NUMBER() OVER (
        PARTITION BY visitorid, session_id
        ORDER BY sequence_no
    ) AS event_pos_in_session
FROM sessionized
ORDER BY timestamp, visitorid, sequence_no;
"""

In [5]:
# events_df with sessions
sessionized_events_df = duckdb.query(sessionization_query).to_df()

all_cols = list(sessionized_events_df.columns) 
all_cols.remove('prev_ts')

sessionized_events_df = sessionized_events_df[all_cols]

# splitting for train and test time-wise
session_train, session_test = train_test_split(sessionized_events_df, test_size=0.4, shuffle=False)

# storing ordered transactions df (it will be sorted since sessionized_df is sorted now)
orders_train = session_train[session_train.transactionid.notnull()]
orders_test = session_test[session_test.transactionid.notnull()]

# resetting indices
session_train.reset_index(drop=True, inplace=True)
session_test.reset_index(drop=True, inplace=True)
orders_train.reset_index(drop=True, inplace=True)
orders_test.reset_index(drop=True, inplace=True)

### CG evaluation 

In [6]:
# since the events df is now sorted, lets take out 10000 random samples from the df 

# we will exclude the 1st event of every user's life so that we atleast have 1 data point to get the recommendations 
first_session_first_event = ~((session_test.session_id == 1) & (session_test.is_new_session == 1))

# this consists 10000 events with atleast 1 previous interaction of user
rows = 20000
query_base = session_test.loc[(first_session_first_event & (session_test.index <= rows))]

In [7]:
# query table for past data

def build_query_table_past(query_df, session_past_df, k=10):

    output_rows = []

    # group once for speed
    user_groups = {
        user_id: grp.sort_values("sequence_no").reset_index(drop=True)
        for user_id, grp in session_past_df.groupby("visitorid")
    }

    for query_id, (_, row) in enumerate(query_df.iterrows()):

        # new user in test set
        if row["visitorid"] not in user_groups:

            output_rows.append({
            "query_id": query_id,
            "visitorid": row["visitorid"],
            "session_id": row["session_id"],
            "sequence_no": row["sequence_no"],
            "event_pos_in_session": row["event_pos_in_session"],
            "query_time": row["timestamp"],
            "current_event": row["event"],
            "last_k_items": [],
            "pre_purchased": set(),
            "past_view_count": 0,
            "past_atc_count": 0,
            "past_order_count": 0
            })
            continue

        hist_df = user_groups[row["visitorid"]]

        last_k_items = hist_df.tail(k)["itemid"].tolist()

        event_grouped = hist_df.groupby("event")

        past_events = event_grouped.size()

        # purchased items
        pre_purchased = set(event_grouped['itemid'].apply(list).get('transaction',[]))

        output_rows.append({
            "query_id": query_id,
            "visitorid": row["visitorid"],
            "session_id": row["session_id"],
            "sequence_no": row["sequence_no"],
            "event_pos_in_session": row["event_pos_in_session"],
            "query_time": row["timestamp"],
            "current_event": row["event"],
            "last_k_items": last_k_items,
            "pre_purchased": pre_purchased,
            "past_view_count": past_events.get("view", 0),
            "past_atc_count": past_events.get("addtocart", 0),
            "past_order_count": past_events.get("transaction", 0)
        })

    return pd.DataFrame(output_rows)

In [8]:
def build_query_table_fut(query_df, session_fut_df):

    output_rows = []

    # group once for speed - past data
    user_groups = {
        user_id: grp.sort_values("sequence_no").reset_index(drop=True)
        for user_id, grp in session_fut_df.groupby("visitorid")
    }

    for query_id, (_, row) in enumerate(query_df.iterrows()):

        user_df = user_groups[row["visitorid"]]

        curr_sess = row["session_id"]
        curr_ts = row["timestamp"]

        # future positives within same session, using only future/current views - this should be done with test_set
        fut_df = user_df[
            (user_df["session_id"] == curr_sess) &
            (user_df["timestamp"] >= curr_ts) &
            (user_df["event"] == "view")
        ]

        future_positives = list(dict.fromkeys(fut_df["itemid"].tolist()))

        if len(future_positives) == 0:
            continue

        output_rows.append({
            "query_id": query_id,
            "future_positives": future_positives
        })

    return pd.DataFrame(output_rows)

In [9]:
# remove duplicates basis ('visitorid','session_id','timestamp') so that we keep only one entry per combination for testing
query_base.drop_duplicates(subset=['visitorid','session_id','timestamp'],inplace=True)

query_ip_past = build_query_table_past(query_base, session_train) #for building past based inputs
query_ip_fut = build_query_table_fut(query_base, session_test) #for building future positives

#merging togethor
query_input = pd.merge(left=query_ip_past, right=query_ip_fut, on='query_id', how='inner')

#### Evaluation on basic CGs

In [10]:
# Evaluation with HitRate, Recall, precision and MRR

def evalualtion_cg_query(true_op, pred_op, k):

    true_op_set, pred_op_set = set(true_op), set(pred_op[:k])

    common_items = true_op_set.intersection(pred_op_set)

    # hitrate
    hit_rate_k = 1 if len(common_items) > 0 else 0
    # recall
    recall_k = len(common_items)/len(true_op_set)
    # precision
    precision_k = len(common_items)/len(pred_op_set)
    #  MRR_k
    MRR_k = 0
    for rank, item in enumerate(pred_op[:k], start=1):
        if item in true_op_set:
            MRR_k = 1/rank
            break

    return {"hitrate_k":hit_rate_k, "recall_k":recall_k, "precision_k":precision_k, "MRR_k":MRR_k}

In [11]:
def evalualtion_cg(eval_df, k):

    metrics = defaultdict(list)
    len_df = eval_df.shape[0]

    for _, row in eval_df.iterrows():
        true_op = row['future_positives']
        pred_op = row['predicted_items']

        res = evalualtion_cg_query(true_op, pred_op, k)

        metrics['hitrate_k'].append(res['hitrate_k'])
        metrics['recall_k'].append(res['recall_k'])
        metrics['precision_k'].append(res['precision_k'])
        metrics['MRR_k'].append(res['MRR_k'])

    return {
        'mean_hitrate_k': sum(metrics['hitrate_k'])/len_df,
        'mean_recall_k': sum(metrics['recall_k'])/len_df,
        'mean_precision_k': sum(metrics['precision_k'])/len_df,
        'mean_MRR_k': sum(metrics['MRR_k'])/len_df
    }

### Basic CGs

#### Popularity CG - User Agnostic

In [12]:
# returns top n platform items from past data - user agnostic

def popularity_cg_base(size, orders_past_df):

    return orders_past_df.groupby('itemid').size().sort_values(ascending=False).index.tolist()[:size]

#### Co-Event CG - user specific

In [13]:
# event_type : orders - returns visitorid and items that user bought over last 90 days - for co-order
# event_type : interactions - returns visitorid, session id, min timestamp and items in that session in list over last 30 days - for co-interacted

def build_event_item_table(events_df, event_type, d):

    max_time = events_df.timestamp.max()
    min_time = max_time - pd.DateOffset(days=d)

    if event_type=='orders':
        group = ["visitorid"]
        condition = (events_df.timestamp >= min_time)
        agg_condition = {
            "items": (
                "itemid",
                lambda x: list(dict.fromkeys(x.tolist()))
            )
        }
    
    else:
        group = ["visitorid", "session_id"]
        condition = ((events_df.timestamp >= min_time) & (events_df["event"].isin(["view", "addtocart"])))
        agg_condition = {
            "session_start": ("timestamp", "min"),

            "items": (
                "itemid",
                lambda x: list(dict.fromkeys(x.tolist()))
            )
        }

    browse_df = events_df.loc[condition].copy()

    event_items = (
        browse_df.groupby(group)
        .agg(**agg_condition)
        .reset_index()
    )

    return event_items

In [14]:
# item item co-event count stored 

def build_same_event_item_map(event_item_df, event_type, top_m, max_items_per_session=30):

    item_to_related = defaultdict(Counter)

    for items in event_item_df["items"]:
        if len(items) < 2:
            continue

        # optional cap to avoid huge sessions causing too much work
        if event_type == 'interactions' and len(items) > max_items_per_session:
            items = items[:max_items_per_session]

        for i, item in enumerate(items):
            other_items = items[:i] + items[i+1:]
            item_to_related[item].update(other_items)

    # keep only top M related items per item for fast query-time retrieval
    item_to_top_related = {
        item: counter.most_common(top_m)
        for item, counter in item_to_related.items()
    }

    return item_to_top_related

In [ ]:
def co_event_cg(input_q, size, item_to_top_related):

    '''
    inputs - 
    input_q - pre-purchased list, last k interacted items (click/atc/order), past popular items of platform that the user hasn't bought yet
    size - size of output
    item_to_top_related - co-order map or co-interaction map

    output -
    recommender_list - array of recommended products
    '''

    already_purchased = input_q["pre_purchased"]
    pre_item_list = input_q["last_k_items"]
    popular_items = input_q["pred_popularity_cg"]

    if not pre_item_list:
        return popular_items[:size]

    # keep unique recent items, but preserve order
    seed_items = list(dict.fromkeys(pre_item_list))

    scores = Counter()

    # optional: give more weight to more recent items
    for rank, seed_item in enumerate(reversed(seed_items), start=1):
        weight = 1.0 / rank

        for related_item, cnt in item_to_top_related.get(seed_item, []):
            if related_item != seed_item and related_item not in already_purchased:
                scores[related_item] += weight * cnt

    recs = [item for item, _ in scores.most_common(math.ceil(1.5 * size))]
    rec_set = set(recs)

    if len(recs) < size:
        for item in popular_items:
            if item not in rec_set and item not in already_purchased:
                recs.append(item)
                rec_set.add(item)
            if len(recs) >= size:
                break

    return recs[:size]

### Evaluation

#### Pre-Computations

In [16]:
# store popular items
popular_items = popularity_cg_base(size=100, orders_past_df=orders_train)

# returns popular items that are not yet bought 
def get_popular_reco(pre_purchased, popular_items):

    return [item for item in popular_items if item not in pre_purchased]

In [17]:
# popularity cg output
popularity_op_df = pd.DataFrame(
    {'future_positives': query_input.future_positives, 
     'predicted_items': query_input['pre_purchased'].apply(get_popular_reco, popular_items=popular_items)
    })

query_input['pred_popularity_cg'] = popularity_op_df['predicted_items']

##### Important co-event maps

In [18]:
# creating previous order mappings for quick retreival

user_orders_df = build_event_item_table(orders_train, event_type='orders', d=90)
item_to_top_ordered_map = build_same_event_item_map(user_orders_df, event_type='orders', top_m=150, max_items_per_session=30)

In [19]:
# creating previous interaction mappings for quick retreival

session_items_df = build_event_item_table(session_train, event_type='interactions', d=30)
item_to_top_interacted_map = build_same_event_item_map(session_items_df, event_type='interactions', top_m=300, max_items_per_session=30)

#### Get Co-Event CG recommendations

In [32]:
# function to get CG output from co-order cg
def get_co_order_reco(query_input, size=100):
    return query_input[['pre_purchased','last_k_items','query_time','pred_popularity_cg']].apply(co_event_cg, axis=1, size=size, item_to_top_related=item_to_top_ordered_map)

In [37]:
# function to get CG output from co-interaction cg
def get_co_interaction_reco(query_input, size=100):
    return query_input[['pre_purchased','last_k_items','query_time','pred_popularity_cg']].apply(co_event_cg, axis=1, size=size, item_to_top_related=item_to_top_interacted_map)

In [40]:
# co-order cg output
co_order_op_df = pd.DataFrame(
    {'future_positives': query_input.future_positives, 
     'predicted_items': get_co_order_reco(query_input)
    })

query_input['pred_co_order_cg'] = co_order_op_df['predicted_items']

In [41]:
# co-interaction cg output
co_interaction_op_df = pd.DataFrame(
    {'future_positives': query_input.future_positives, 
     'predicted_items': get_co_interaction_reco(query_input)
    })

query_input['pred_co_interaction_cg'] = co_interaction_op_df['predicted_items']

In [42]:
print("popularity CG evalutaion : ",evalualtion_cg(popularity_op_df, k=100))

print("Co-order CG evalutaion : ",evalualtion_cg(co_order_op_df, k=100))

print("Co-interaction CG evalutaion : ",evalualtion_cg(co_interaction_op_df, k=100))

popularity CG evalutaion :  {'mean_hitrate_k': 0.07368311036789298, 'mean_recall_k': 0.02338048637693185, 'mean_precision_k': 0.001031520263597404, 'mean_MRR_k': 0.012425734721438916}
Co-order CG evalutaion :  {'mean_hitrate_k': 0.06093227424749164, 'mean_recall_k': 0.01939602838455705, 'mean_precision_k': 0.0006912988948669398, 'mean_MRR_k': 0.008039309081917773}
Co-interaction CG evalutaion :  {'mean_hitrate_k': 0.1519648829431438, 'mean_recall_k': 0.10200286778641476, 'mean_precision_k': 0.0021258361204013437, 'mean_MRR_k': 0.0395496037685434}


### Storing the data sets

In [43]:
# saving the tables back
input_cols = [col for col in query_input.columns if col not in {'pred_popularity_cg', 'pred_co_order_cg', 'pred_co_interaction_cg'}]

# overall data
# df_cat.to_csv('data/intermediate_files/category_input.csv', index=False)
sessionized_events_df.to_pickle('data/intermediate_files/sessionized_events.pkl')
df_item_prop.to_pickle('data/intermediate_files/item_properties.pkl')

# transformed data
session_train.to_pickle('data/intermediate_files/session_train.pkl')
session_test.to_pickle('data/intermediate_files/session_test.pkl')
orders_train.to_pickle('data/intermediate_files/orders_train.pkl')
orders_test.to_pickle('data/intermediate_files/orders_test.pkl')

# query inputs & outputs
query_input[input_cols].to_pickle('data/intermediate_files/cg_input_query_10k.pkl')
query_input.to_pickle('data/intermediate_files/cg_ip_op_query_10k.pkl')

### Other Queries

In [ ]:
# # Any time a user comes onto the platform, we check the user's last interaction of item, 
# # and recommend products that other users bought who also bought the current user's last interacted item.
# # if no user bought the last interacted item or if user is new and no prior data is available, we recommend top 10 platform popular item

# def co_order_cg(input_q, size, orders_past_df):
    
#     '''
#     inputs - 
#     input_q - pre-purchased list, last k interacted items (click/atc/order), query time
#     size - size of output
#     popular items - past popular items of platform that the user hasn't bought yet

#     output -
#     recommender_list - array of recommended products
#     '''

#     # get relevant inputs
#     already_purchased = input_q['pre_purchased']
#     pre_item_list = input_q['last_k_items']
#     popular_pers = input_q['pred_popularity_cg']
    
#     # if user has not interacte ever
#     if not pre_item_list:
#         return popular_pers[:size]
    
#     # get its prev interactions
#     prev_item_ids = set(pre_item_list)

#     # get other users co-order lists
#     prev_purchased_items = orders_past_df.groupby('visitorid')['itemid'].apply(list).tolist()

#     # count co-occurences
#     item_counter = Counter()

#     for prev_item_id in prev_item_ids:
#         for purchase_list in prev_purchased_items:
#             p_set = set(purchase_list) 
#             if prev_item_id in p_set:
#                 item_counter.update(p_set - {prev_item_id})

#     # no one bought the any of the interacted items yet
#     if not item_counter: 
#         return popular_pers[:size]

#     recommender_list = [item for item, _ in item_counter.most_common(math.ceil(1.5*size)) if item not in already_purchased]
#     recommender_set = set(recommender_list)

#     # if recommendation list length < size, we add popular items as well
#     if len(recommender_list) < size:
#         for item in popular_pers:
#             if item not in recommender_set:
#                 recommender_list.append(item)
#                 recommender_set.add(item)

#     return recommender_list[:size]

In [ ]:
# older query table - not wrong though, needs slight changes and inefficent

# def build_query_table(df, session_df):

#     query_id = 0
#     output_df = []
    
#     for _, row in df.iterrows():
#         new_row = []

#         new_row.append(query_id)                               #query_id
#         new_row.append(row['visitorid'])                       #visitorid
#         new_row.append(row['session_id'])                      #session_id
#         new_row.append(row['timestamp'])                       #query_time

#         user_df = session_df.loc[(session_df.visitorid == row['visitorid'])]

#         # k = 10
#         pre_item_list = user_df.loc[(user_df.timestamp < row['timestamp'])].nlargest(10, 'timestamp')['itemid'].to_list()
#         new_row.append(pre_item_list)                          #last k items - can be empty as well

#         past_events = user_df.loc[(user_df.timestamp < row['timestamp'])].groupby('event').size()
        
#         new_row.append(past_events.get('view', 0))             #past view count
#         new_row.append(past_events.get('addtocart', 0))        #past atc count
#         new_row.append(past_events.get('transaction', 0))      #past order count

        
#         future_positives = user_df.loc[(user_df.timestamp >= row['timestamp']) & (user_df.session_id == row['session_id']),'itemid'].to_list()
#         new_row.append(list(dict.fromkeys(future_positives)))  # future positives

#         output_df.append(new_row)
#         query_id += 1

#     new_table_columns = ['query_id', 'visitorid', 'session_id', 'query_time', 'last_k_items', 'past_view_count', 'past_atc_count', 'past_order_count', 'future_positives']
#     query_table = pd.DataFrame(output_df, columns= new_table_columns)

#     return query_table
